### Q. What is Batch And Streaming Data Processing?

#### 1) Batch Processing
**Batch processing** refers to processing data in large, fixed-size groups (batches) at scheduled intervals.

**Characteristics:**
- Data is collected over time and processed together
- Runs on a schedule (hourly, daily, weekly, etc.)
- Processes historical data
- Higher latency (minutes to hours)
- More efficient for large volumes of data
- Simpler to implement and debug

**Use Cases:**
- End-of-day reporting
- Monthly sales analysis
- Historical data analysis
- ETL jobs that run nightly

**Example in Databricks:**
```python
# Batch: Read entire table at once
df = spark.read.table("sales_data")
df.groupBy("product").sum("revenue").write.mode("overwrite").saveAsTable("daily_revenue")
```

---

#### 2) Streaming Processing
**Streaming processing** (also called stream processing or real-time processing) continuously processes data as it arrives.

**Characteristics:**
- Data is processed record-by-record or in micro-batches as it arrives
- Runs continuously
- Processes real-time or near-real-time data
- Low latency (seconds or less)
- More complex to implement
- Ideal for time-sensitive use cases

**Use Cases:**
- Real-time fraud detection
- IoT sensor monitoring
- Live dashboard updates
- Click-stream analysis
- Stock market monitoring

**Example in Databricks:**
```python
# Streaming: Continuously read new data
df = spark.readStream.table("sales_data")
df.groupBy("product").sum("revenue").writeStream.outputMode("complete").toTable("live_revenue")
```

<br><img src="images/image_1788690553999.png" width="800">

---

#### Key Differences

| Aspect | Batch | Streaming |
|--------|-------|----------|
| **Data Processing** | All at once | Continuous |
| **Latency** | Minutes to hours | Seconds to milliseconds |
| **Data Scope** | Historical, bounded | Real-time, unbounded |
| **Complexity** | Lower | Higher |
| **Cost** | Lower (intermittent resources) | Higher (always running) |
| **Use When** | Latency not critical | Immediate insights needed |

### 📋 Finguard Streaming Project Overview
![image_1789227808599.png](images/image_1789227808599.png "image_1789227808599.png")
![image_1789227793709.png](images/image_1789227793709.png "image_1789227793709.png")

---

#### 🏗️ Architecture Components

![image_1789732421639.png](images/image_1789732421639.png "image_1789732421639.png")

##### 1️⃣ Data Sources

**Streaming Sources** (Real-time):
* **Kafka Topics**: Live transaction events from payment gateways
* **JSON Files**: Streaming transaction logs from ATMs and POS terminals

**Batch Sources** (Reference Data):
* **PostgreSQL Database**: Customer profiles, account details, merchant information, and historical patterns

##### 2️⃣ Ingestion Layer

* **Spark Structured Streaming**: Consumes and processes Kafka streams in real-time
* **Auto Loader**: Incrementally ingests streaming JSON files with automatic schema evolution
* **Lakeflow Connect**: Manages batch ingestion from PostgreSQL for reference data

##### 3️⃣ Processing Layer: Lakeflow Declarative Pipelines

Implements the **Medallion Architecture** (Bronze → Silver → Gold) using streaming tables:

**Bronze Layer (Raw)**
* Ingests raw streaming data with minimal transformation
* Preserves data lineage and supports reprocessing
* Handles schema evolution automatically

**Silver Layer (Enriched)**
* Cleanses and standardizes data quality
* Applies **stream-stream joins** between live transactions and fraud watchlists
* Performs **stream-static joins** with customer profiles and merchant reference data
* Implements data validation and deduplication logic

**Gold Layer (Curated)**
* Executes fraud detection algorithms and business rules
* Calculates transaction risk scores using multiple indicators
* Aggregates patterns for anomaly detection (velocity checks, geographic anomalies)
* Produces analytics-ready datasets for consumption

##### 4️⃣ Consumption Layer

* **Real-time Alerts**: Immediate notifications sent to fraud analysts for high-risk transactions (email, Slack, PagerDuty)
* **Live Dashboards**: Operational fraud monitoring console with real-time metrics and KPIs
* **APIs**: RESTful endpoints for downstream applications

##### 5️⃣ Supporting Infrastructure

* **Orchestration**: Lakeflow Jobs schedule and monitor pipeline execution
* **Governance**: Unity Catalog provides data security, access control, and audit logging
* **Storage**: Delta Lake ensures ACID transactions, time travel, and schema enforcement
* **Monitoring**: Built-in pipeline observability with metrics and alerts

---

#### 🎯 Key Learning Objectives

This project demonstrates critical data engineering concepts:

1. **Streaming vs Batch Processing**: Understanding when to use real-time vs scheduled processing
2. **Spark Structured Streaming**: Building fault-tolerant, scalable streaming pipelines
3. **Auto Loader**: Incrementally processing cloud files as they arrive with schema inference
4. **Stream Joins**: 
   * Stream-stream joins for real-time correlation
   * Stream-static joins for data enrichment
5. **Medallion Architecture**: Organizing data in Bronze-Silver-Gold layers for quality and reusability
6. **Lakeflow Declarative Pipelines**: Infrastructure-as-code approach to data pipelines
7. **Window-Based Analytics**: Time-based aggregations for pattern detection
8. **Real-time Analytics**: Complete data flow from ingestion to consumption in seconds

---


### What is Apache Kafka?

**Apache Kafka** is a distributed streaming platform designed to handle high-throughput, real-time data feeds. Originally developed by LinkedIn and later open-sourced through the Apache Software Foundation, Kafka is widely used for building real-time data pipelines and streaming applications.

**Key Capabilities:**
- **Publish and Subscribe**: Read and write streams of records similar to a message queue
- **Store**: Persist streams of records in a fault-tolerant, durable way
- **Process**: Process streams of records as they occur in real-time

**Common Use Cases:**
- Real-time data pipelines between systems
- Real-time streaming applications that transform or react to data
- Event sourcing architectures
- Log aggregation
- Metrics collection and monitoring
- Click-stream analysis
- IoT sensor data ingestion

---

### Kafka Architecture

<br><img src="images/image_1788694882227.png" width="800">

#### Core Components

**1. Producer**
- Applications that **publish (write) data** to Kafka topics
- Can send records to specific partitions or let Kafka distribute them
- Support batching, compression, and acknowledgment configurations
- Example: Web server logs, IoT sensors, transaction systems

**2. Consumer**
- Applications that **subscribe to (read) data** from Kafka topics
- Organized into **consumer groups** for parallel processing
- Track their position (offset) in each partition
- Can replay messages by resetting offsets
- Example: Stream processing applications, analytics engines, data warehouses

**3. Broker**
- A **Kafka server** that stores data and serves client requests
- Each broker can handle thousands of reads/writes per second
- A Kafka cluster typically consists of multiple brokers for scalability and fault tolerance
- Brokers are stateless; they rely on ZooKeeper (or KRaft) for coordination

**4. Topic**
- A **category or feed name** to which records are published
- Topics are divided into **partitions** for parallelism and scalability
- Each record in a partition has a unique sequential ID called an **offset**
- Retention policy determines how long data is kept (time-based or size-based)

**5. Partition**
- A **division of a topic** that allows parallel processing
- Each partition is an ordered, immutable sequence of records
- Partitions are replicated across brokers for fault tolerance
- Data within a partition is ordered, but not across partitions

**6. ZooKeeper (or KRaft)**
- **ZooKeeper**: Coordinates the Kafka cluster, manages broker membership, leader election, and metadata
- **KRaft** (Kafka Raft): The new consensus protocol that removes ZooKeeper dependency (Kafka 3.0+)
- Stores configuration, topic metadata, partition leadership, and consumer group offsets

---

#### Key Architectural Concepts

**Replication**
- Each partition has one **leader** and multiple **followers** (replicas)
- All reads and writes go through the leader
- Followers replicate data from the leader for fault tolerance
- If the leader fails, a follower is elected as the new leader
- **Replication factor** determines how many copies of data exist

**Consumer Groups**
- Multiple consumers can form a group to share the workload
- Each partition is consumed by exactly one consumer in the group
- Enables horizontal scalability: add more consumers to process data faster
- Different consumer groups can independently consume the same topic

**Offsets**
- A unique sequential number assigned to each record in a partition
- Consumers track their position using offsets
- Offsets are stored in a special Kafka topic (`__consumer_offsets`)
- Enables replay, exactly-once semantics, and fault recovery

**Log Retention**
- Kafka retains messages for a configurable period (default 7 days)
- Retention can be time-based or size-based
- Old data is deleted or compacted based on the retention policy
- Enables "time travel" — consumers can read historical data

---

### Kafka in the FinGuard Project

- **Kafka Topics** contain live credit card transaction streams from different sources.
- **Spark Structured Streaming** acts as the consumer, continuously reading from Kafka.
- Each transaction is processed in real-time for fraud detection.
- Low-latency ingestion enables detection within seconds.

```python
# Example: Reading from Kafka in Databricks
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "broker1:9092,broker2:9092") \
    .option("subscribe", "credit_card_transactions") \
    .option("startingOffsets", "latest") \
    .load()

# Kafka value is binary; decode it
transactions = df.selectExpr("CAST(value AS STRING) as json") \
    .select(from_json(col("json"), schema).alias("data")) \
    .select("data.*")
```

---
